In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer,AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
df=pd.read_csv('Pseudo_review_label_checked.csv')

df=df.drop(columns=['product_name','confidence_score'])
print(df.info())
print(df.head())

In [ ]:
df['labels']=df['labels'].map({'negative': 0, 'neutral': 1, 'positive': 2})
print(df['labels'].value_counts())
print(df.head())

In [ ]:
train_df,test_df=train_test_split(df,test_size=0.2,random_state=42,stratify=df['labels'])


In [ ]:
train_set=Dataset.from_pandas(train_df)
test_set=Dataset.from_pandas(test_df)

train_set = train_set.remove_columns(["__index_level_0__"])
test_set = test_set.remove_columns(["__index_level_0__"])
print(train_set)
print(test_set)

In [ ]:
tokenizer=AutoTokenizer.from_pretrained('cardiffnlp/twitter-roberta-base-sentiment-latest')

In [ ]:
def tokenize_function(example):
    return tokenizer(example["review_summary"], truncation=True, padding="max_length", max_length=124)

train_set=train_set.map(tokenize_function, batched=True)
test_set=test_set.map(tokenize_function, batched=True)


In [ ]:
model=AutoModelForSequenceClassification.from_pretrained('cardiffnlp/twitter-roberta-base-sentiment-latest',num_labels=3)

training_args=TrainingArguments(output_dir='./results',
                               num_train_epochs=2,
                               per_device_train_batch_size=16,
                                per_device_eval_batch_size=16,
                                eval_strategy="epoch")
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_set,
    eval_dataset=test_set
)


In [ ]:
print("start")
trainer.train()
print("done")

In [ ]:
predictions = trainer.predict(test_set)
preds = predictions.predictions.argmax(axis=-1)
labels = predictions.label_ids


print(accuracy_score(labels, preds))
print(classification_report(labels, preds))

In [ ]:
model.save_pretrained("model_sentiment")
tokenizer.save_pretrained("model_tokenizer")